[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/11.%20The%20Dynamic%20Berth%20Allocation%20Problem/P11-Tier-3_executed.ipynb) [![Open In SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 11. The Dynamic Berth Allocation Problem
## Tier 3 — A Genetic Algorithm Heuristic

### Goal and learning objectives

This notebook uses a genetic algorithm (GA) to construct berth schedules for a small dynamic berth allocation problem. The example demonstrates how candidate schedules are represented, evaluated, selected, recombined, mutated, and repaired. A GA is a stochastic heuristic: it searches for good schedules but does not certify that a schedule is globally optimal.

By the end, you should be able to trace how the vessel data become chromosomes, how the fitness function ranks chromosomes, and how each genetic operator contributes to the search.

### Problem inputs and output

The input consists of vessels with arrival times, desired departure times, lengths, TEU capacities, and berth-specific processing times, together with berths and their length capacities. The output is a schedule specifying a berth and start time for each assigned vessel, along with service-time and waiting-time metrics.

The example contains **3 vessels and 2 berths**. It is a compact demonstration of the algorithm, not a 10-vessel benchmark.

### Model assumptions and scope

- A vessel occupies one berth continuously from its start time until its processing is complete.
- A vessel may use a berth only if its length does not exceed that berth's capacity. Processing time depends on the vessel–berth pair.
- A vessel cannot start before its arrival. Berth schedules must not overlap; intervals are treated as half-open, so one vessel may start exactly when another finishes.
- The desired departure time is reported as a deadline for measuring delay. It is **not** a hard feasibility constraint and does not enter the fitness function.
- TEU capacity is descriptive data and is not used by this scheduling model. The model also omits spatial berth positions, tides, setup times, and uncertainty in arrivals or processing times.

### Chromosome and fitness

A chromosome is a list of genes of the form $(v,b,s)$: vessel identifier $v$, berth identifier $b$, and proposed start time $s$. The list is initially generated from a shuffled vessel order. During repair, genes are processed by proposed start time and their start times are adjusted to produce a non-overlapping schedule.

For a valid assignment, vessel $v$ has arrival time $a_v$, processing time $p_{vb}$ on berth $b$, and start time $s_v$. Its waiting time is $s_v-a_v$, and its service time is waiting plus processing. The total service time is therefore

$$C = \sum_{v\ \mathrm{assigned}} \bigl((s_v-a_v)+p_{v b_v}\bigr).$$

The fitness used by the code is

$$F = \frac{1}{1+C+P} + 100A,$$

where $A$ is the number of assigned vessels and $P$ is the accumulated penalty for invalid assignments or conflicts. Fitness is maximized. The assignment bonus makes assigning more vessels the primary priority; among schedules with the same number assigned, the reciprocal term favors lower service time and fewer penalties. The factor 100 and the penalty values are heuristic weights, not physical quantities.

### Genetic search, step by step

1. **Initialization:** generate a population of candidate chromosomes using shuffled vessel orders and compatible berths.
2. **Evaluation:** calculate each chromosome's fitness.
3. **Selection:** tournament selection samples candidates and chooses the fittest sampled candidate as a parent.
4. **Crossover:** order crossover copies a segment from one parent and fills the remaining positions using the other parent's vessel order.
5. **Mutation:** with the specified probability, one vessel may be moved to a different compatible berth.
6. **Repair:** reconstruct feasible berth timing by processing genes in proposed-start order and moving each start to the earliest time allowed by arrival and berth availability.
7. **Elitism and repetition:** retain the best candidates and repeat the search for the requested number of generations.

For $n$ vessels and population size $N$, evaluating one population takes $O(Nn^2)$ time in this implementation because overlap checks compare assignments on each berth. Across $G$ generations, the dominant evaluation work is $O(GNn^2)$. The GA does not enumerate or guarantee the best schedule; its result depends on stochastic choices and the chosen parameters.

### Interpreting results

The reported fitness is a search score, not a number of hours. Use the schedule and service-time metrics to interpret operational performance. The comparison section reports a simple earliest-deadline-first (EDF) baseline for context; a single stochastic run does not establish that the GA is generally better. Execution times are machine- and run-dependent, especially for this small example.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass
import random
import time
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Define the vessels and berths

Time values are in hours and length capacities are in metres. Processing times are indexed by berth identifier. Keeping these data explicit makes it possible for the chromosome generator, repair routine, fitness calculation, and result reporting to use the same problem definition.

In [ ]:
@dataclass
class Vessel:
    """Input data describing one vessel."""

    id: int
    arrival_time: int
    desired_departure: int
    length: int
    teu_capacity: int
    processing_times: Dict[int, int]


@dataclass
class Berth:
    """Input data describing one berth."""

    id: int
    capacity: int


vessels = [
    Vessel(1, 0, 10, 200, 5000, {1: 4, 2: 5}),
    Vessel(2, 2, 12, 250, 8000, {1: 3, 2: 4}),
    Vessel(3, 4, 15, 300, 12000, {1: 6, 2: 5}),
]

berths = [
    Berth(1, 350),
    Berth(2, 400),
]

print(f"Problem: {len(vessels)} vessels, {len(berths)} berths")
for vessel in vessels:
    print(
        f"  Vessel {vessel.id}: TEU={vessel.teu_capacity}, "
        f"arrival={vessel.arrival_time} h, "
        f"desired departure={vessel.desired_departure} h, "
        f"length={vessel.length} m"
    )
for berth in berths:
    print(f"  Berth {berth.id}: capacity={berth.capacity} m")

## 2. Implement the genetic algorithm

The methods below correspond to the stages described above. The representation stores a vessel's berth and proposed start alongside its identifier, so crossover and mutation operate on complete vessel genes. The repair step then makes the timing consistent with arrival times and berth availability before fitness is used to compare candidate schedules.

In [ ]:
class GeneticAlgorithmDBAPSolver:
    """Genetic-algorithm heuristic for the dynamic berth allocation problem."""

    def __init__(
        self,
        vessels: List[Vessel],
        berths: List[Berth],
        pop_size: int = 50,
        generations: int = 100,
        crossover_rate: float = 0.8,
        mutation_rate: float = 0.15,
        tournament_size: int = 3,
        elitism_count: int = 2,
    ):
        if pop_size < 2:
            raise ValueError("Population size must be at least 2.")
        if generations < 1:
            raise ValueError("The number of generations must be positive.")
        if not 1 <= tournament_size <= pop_size:
            raise ValueError("Tournament size must be between 1 and population size.")
        if not 0 <= elitism_count <= pop_size:
            raise ValueError("Elitism count must be between 0 and population size.")
        if not 0 <= crossover_rate <= 1 or not 0 <= mutation_rate <= 1:
            raise ValueError("Crossover and mutation rates must be between 0 and 1.")

        self.vessels = vessels
        self.berths = berths
        self.vessel_dict = {vessel.id: vessel for vessel in vessels}
        self.berth_dict = {berth.id: berth for berth in berths}

        self.pop_size = pop_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.tournament_size = tournament_size
        self.elitism_count = elitism_count

        self.population = []
        self.fitness_history = []
        self.diversity_history = []
        self.best_solution = None
        self.best_fitness = float("-inf")

    def create_individual(self) -> List[Tuple[int, int, int]]:
        """Create a chromosome by assigning vessels in a shuffled order."""
        vessel_order = random.sample(self.vessels, len(self.vessels))
        berth_availability = {berth.id: 0 for berth in self.berths}
        chromosome = []

        for vessel in vessel_order:
            compatible_berths = [
                berth.id
                for berth in self.berths
                if vessel.length <= berth.capacity
                and berth.id in vessel.processing_times
            ]

            if not compatible_berths:
                chromosome.append((vessel.id, -1, -1))
                continue

            berth_id = random.choice(compatible_berths)
            start_time = max(
                vessel.arrival_time,
                berth_availability[berth_id],
            )
            processing_time = vessel.processing_times[berth_id]
            berth_availability[berth_id] = start_time + processing_time
            chromosome.append((vessel.id, berth_id, start_time))

        return chromosome

    def initialize_population(self) -> None:
        """Generate the first population of candidate schedules."""
        self.population = [
            self.create_individual() for _ in range(self.pop_size)
        ]
        print(f"Initialized population with {self.pop_size} individuals")

    def calculate_fitness(
        self,
        chromosome: List[Tuple[int, int, int]],
    ) -> float:
        """Return a higher-is-better score for a candidate schedule."""
        total_service_time = 0
        penalty = 0
        vessels_assigned = 0
        berth_schedules = {berth.id: [] for berth in self.berths}

        for vessel_id, berth_id, start_time in chromosome:
            vessel = self.vessel_dict.get(vessel_id)
            if vessel is None:
                penalty += 1000
                continue

            if berth_id == -1 or start_time == -1:
                penalty += 1000
                continue

            berth = self.berth_dict.get(berth_id)
            if (
                berth is None
                or vessel.length > berth.capacity
                or berth_id not in vessel.processing_times
            ):
                penalty += 500
                continue

            if start_time < vessel.arrival_time:
                penalty += 200
                continue

            processing_time = vessel.processing_times[berth_id]
            completion_time = start_time + processing_time
            has_overlap = any(
                not (
                    completion_time <= other_start
                    or start_time >= other_completion
                )
                for other_start, other_completion in berth_schedules[berth_id]
            )

            if has_overlap:
                penalty += 300
                continue

            berth_schedules[berth_id].append((start_time, completion_time))
            waiting_time = start_time - vessel.arrival_time
            total_service_time += waiting_time + processing_time
            vessels_assigned += 1

        total_cost = total_service_time + penalty
        assignment_bonus = vessels_assigned * 100
        return 1.0 / (1.0 + total_cost) + assignment_bonus

    def tournament_selection(
        self,
        fitness_values: List[float],
    ) -> List[Tuple[int, int, int]]:
        """Choose the fittest candidate among randomly sampled individuals."""
        tournament_indices = random.sample(
            range(len(self.population)),
            self.tournament_size,
        )
        best_index = max(
            tournament_indices,
            key=lambda index: fitness_values[index],
        )
        return self.population[best_index]

    def order_crossover(
        self,
        parent1: List[Tuple[int, int, int]],
        parent2: List[Tuple[int, int, int]],
    ) -> Tuple[List[Tuple[int, int, int]], List[Tuple[int, int, int]]]:
        """Combine parent vessel orders while retaining complete vessel genes."""
        size = len(parent1)
        if size != len(parent2):
            raise ValueError("Parents must have the same chromosome length.")
        if size < 2:
            return deepcopy(parent1), deepcopy(parent2)

        start, end = sorted(random.sample(range(size), 2))
        offspring1 = [None] * size
        offspring2 = [None] * size
        offspring1[start : end + 1] = parent1[start : end + 1]
        offspring2[start : end + 1] = parent2[start : end + 1]

        def fill_offspring(offspring, segment_parent, order_parent):
            segment_vessels = {
                gene[0]
                for gene in offspring[start : end + 1]
            }
            missing_genes = [
                gene for gene in order_parent
                if gene[0] not in segment_vessels
            ]
            empty_positions = [
                position
                for position, gene in enumerate(offspring)
                if gene is None
            ]
            for position, gene in zip(empty_positions, missing_genes):
                offspring[position] = gene

        fill_offspring(offspring1, parent1, parent2)
        fill_offspring(offspring2, parent2, parent1)
        return offspring1, offspring2

    def mutate(
        self,
        chromosome: List[Tuple[int, int, int]],
    ) -> List[Tuple[int, int, int]]:
        """Move one vessel to a different compatible berth when possible."""
        mutated = deepcopy(chromosome)
        if not mutated:
            return mutated

        mutate_index = random.randrange(len(mutated))
        vessel_id, current_berth, _ = mutated[mutate_index]
        vessel = self.vessel_dict.get(vessel_id)
        if vessel is None:
            return mutated

        compatible_berths = [
            berth.id
            for berth in self.berths
            if vessel.length <= berth.capacity
            and berth.id in vessel.processing_times
            and berth.id != current_berth
        ]
        if compatible_berths:
            new_berth = random.choice(compatible_berths)
            mutated[mutate_index] = (
                vessel_id,
                new_berth,
                vessel.arrival_time,
            )
        return mutated

    def repair_chromosome(
        self,
        chromosome: List[Tuple[int, int, int]],
    ) -> List[Tuple[int, int, int]]:
        """Rebuild berth timing so assignments respect arrivals and capacity."""
        repaired = deepcopy(chromosome)
        repaired.sort(key=lambda gene: gene[2] if gene[2] >= 0 else float("inf"))
        berth_availability = {berth.id: 0 for berth in self.berths}

        for index, (vessel_id, berth_id, _) in enumerate(repaired):
            vessel = self.vessel_dict.get(vessel_id)
            if vessel is None:
                continue

            berth = self.berth_dict.get(berth_id)
            is_compatible = (
                berth is not None
                and vessel.length <= berth.capacity
                and berth_id in vessel.processing_times
            )

            if not is_compatible:
                compatible_berths = [
                    candidate.id
                    for candidate in self.berths
                    if vessel.length <= candidate.capacity
                    and candidate.id in vessel.processing_times
                ]
                if not compatible_berths:
                    repaired[index] = (vessel_id, -1, -1)
                    continue
                berth_id = random.choice(compatible_berths)

            feasible_start = max(
                vessel.arrival_time,
                berth_availability[berth_id],
            )
            repaired[index] = (vessel_id, berth_id, feasible_start)
            berth_availability[berth_id] = (
                feasible_start + vessel.processing_times[berth_id]
            )

        return repaired

    def evolve_generation(self) -> Tuple[float, List[Tuple[int, int, int]]]:
        """Evaluate the population and construct its successor generation."""
        fitness_values = [
            self.calculate_fitness(individual)
            for individual in self.population
        ]
        best_index = max(
            range(len(fitness_values)),
            key=fitness_values.__getitem__,
        )
        current_best_fitness = fitness_values[best_index]
        current_best = self.population[best_index]

        if current_best_fitness > self.best_fitness:
            self.best_fitness = current_best_fitness
            self.best_solution = deepcopy(current_best)

        sorted_indices = sorted(
            range(len(fitness_values)),
            key=lambda index: fitness_values[index],
            reverse=True,
        )
        new_population = [
            deepcopy(self.population[index])
            for index in sorted_indices[: self.elitism_count]
        ]

        while len(new_population) < self.pop_size:
            parent1 = self.tournament_selection(fitness_values)
            parent2 = self.tournament_selection(fitness_values)

            if random.random() < self.crossover_rate:
                offspring1, offspring2 = self.order_crossover(parent1, parent2)
            else:
                offspring1, offspring2 = deepcopy(parent1), deepcopy(parent2)

            if random.random() < self.mutation_rate:
                offspring1 = self.mutate(offspring1)
            if random.random() < self.mutation_rate:
                offspring2 = self.mutate(offspring2)

            new_population.append(self.repair_chromosome(offspring1))
            if len(new_population) < self.pop_size:
                new_population.append(self.repair_chromosome(offspring2))

        self.population = new_population[: self.pop_size]
        return current_best_fitness, current_best

    def solve(self) -> Tuple[List[Tuple[int, int, int]], Dict]:
        """Run the configured genetic search and return its best schedule."""
        self.fitness_history = []
        self.diversity_history = []
        self.best_solution = None
        self.best_fitness = float("-inf")

        print("Genetic Algorithm for the Dynamic Berth Allocation Problem")
        print("=" * 60)
        print(f"Population size: {self.pop_size}")
        print(f"Generations: {self.generations}")
        print(f"Crossover rate: {self.crossover_rate}")
        print(f"Mutation rate: {self.mutation_rate}")

        self.initialize_population()
        for generation in range(self.generations):
            best_fitness, _ = self.evolve_generation()
            self.fitness_history.append(best_fitness)
            unique_individuals = {
                tuple(individual) for individual in self.population
            }
            self.diversity_history.append(
                len(unique_individuals) / len(self.population)
            )

            if generation % 10 == 0 or generation == self.generations - 1:
                average_fitness = np.mean(
                    [self.calculate_fitness(individual) for individual in self.population]
                )
                print(
                    f"Generation {generation:3d}: "
                    f"Best fitness = {best_fitness:.4f}, "
                    f"Average fitness = {average_fitness:.4f}"
                )

        detailed_solution = self.chromosome_to_solution(self.best_solution)
        metrics = self.calculate_solution_metrics(detailed_solution)
        print("\nSearch completed.")
        print(f"Best fitness: {self.best_fitness:.4f}")
        print(f"Total service time: {metrics['total_service_time']:.1f} hours")
        print(
            f"Vessels assigned: {metrics['vessels_assigned']}/{len(self.vessels)}"
        )
        return self.best_solution, metrics

    def chromosome_to_solution(
        self,
        chromosome: List[Tuple[int, int, int]],
    ) -> List[Dict]:
        """Convert chromosome genes into readable schedule records."""
        solution = []
        if chromosome is None:
            return solution

        for vessel_id, berth_id, start_time in chromosome:
            vessel = self.vessel_dict.get(vessel_id)
            if vessel is None or berth_id == -1 or start_time == -1:
                continue

            processing_time = vessel.processing_times[berth_id]
            completion_time = start_time + processing_time
            waiting_time = start_time - vessel.arrival_time
            delay = max(0, completion_time - vessel.desired_departure)
            solution.append(
                {
                    "vessel_id": vessel_id,
                    "berth_id": berth_id,
                    "arrival_time": vessel.arrival_time,
                    "deadline": vessel.desired_departure,
                    "start_time": start_time,
                    "completion_time": completion_time,
                    "processing_time": processing_time,
                    "waiting_time": waiting_time,
                    "delay_beyond_deadline": delay,
                    "service_time": waiting_time + processing_time,
                }
            )
        return solution

    def calculate_solution_metrics(self, solution: List[Dict]) -> Dict:
        """Summarize service, waiting, processing, and deadline metrics."""
        if not solution:
            return {
                "total_service_time": 0,
                "total_waiting_time": 0,
                "total_processing_time": 0,
                "vessels_assigned": 0,
                "vessels_delayed": 0,
                "avg_waiting_time": 0,
                "max_waiting_time": 0,
            }

        frame = pd.DataFrame(solution)
        return {
            "total_service_time": frame["service_time"].sum(),
            "total_waiting_time": frame["waiting_time"].sum(),
            "total_processing_time": frame["processing_time"].sum(),
            "vessels_assigned": len(frame),
            "vessels_delayed": int((frame["delay_beyond_deadline"] > 0).sum()),
            "avg_waiting_time": frame["waiting_time"].mean(),
            "max_waiting_time": frame["waiting_time"].max(),
        }


print("GA solver class defined successfully")

## 3. Run the search and inspect the schedule

The parameter values below define the population size, number of generations, crossover and mutation probabilities, tournament size, and number of elite individuals retained. No random seed is set, so separate runs may produce different schedules and runtimes. The checks verify that the returned schedule assigns each vessel at most once, respects arrivals and berth capacities, has no berth overlaps, and has consistent service-time totals.

In [ ]:
ga_solver = GeneticAlgorithmDBAPSolver(
    vessels=vessels,
    berths=berths,
    pop_size=50,
    generations=100,
    crossover_rate=0.8,
    mutation_rate=0.15,
    tournament_size=3,
    elitism_count=2,
)

start_time = time.perf_counter()
best_solution, metrics = ga_solver.solve()
computation_time = (time.perf_counter() - start_time) * 1000

detailed_solution = ga_solver.chromosome_to_solution(best_solution)
df_solution = pd.DataFrame(detailed_solution)

assert len(detailed_solution) == metrics["vessels_assigned"]
assert len({row["vessel_id"] for row in detailed_solution}) == len(detailed_solution)
for row in detailed_solution:
    vessel = ga_solver.vessel_dict[row["vessel_id"]]
    berth = ga_solver.berth_dict[row["berth_id"]]
    assert row["start_time"] >= vessel.arrival_time
    assert vessel.length <= berth.capacity

for berth_id in ga_solver.berth_dict:
    berth_rows = sorted(
        (row for row in detailed_solution if row["berth_id"] == berth_id),
        key=lambda row: row["start_time"],
    )
    for first, second in zip(berth_rows, berth_rows[1:]):
        assert first["completion_time"] <= second["start_time"]

if detailed_solution:
    assert np.isclose(
        metrics["total_service_time"],
        metrics["total_waiting_time"] + metrics["total_processing_time"],
    )

print(f"\nComputation time: {computation_time:.2f} ms")
print("\nFinal schedule")
print("=" * 30)
for row in sorted(detailed_solution, key=lambda item: item["start_time"]):
    status = (
        "ON TIME"
        if row["delay_beyond_deadline"] == 0
        else f"LATE by {row['delay_beyond_deadline']} h"
    )
    print(
        f"Vessel {row['vessel_id']}: Berth {row['berth_id']}, "
        f"start {row['start_time']} h, complete {row['completion_time']} h, "
        f"wait {row['waiting_time']} h, process {row['processing_time']} h — {status}"
    )

print("\nPerformance metrics")
print(f"Total service time: {metrics['total_service_time']:.1f} hours")
print(f"Total waiting time: {metrics['total_waiting_time']:.1f} hours")
print(f"Total processing time: {metrics['total_processing_time']:.1f} hours")
print(f"Average waiting time: {metrics['avg_waiting_time']:.2f} hours")
print(f"Vessels assigned: {metrics['vessels_assigned']}/{len(vessels)}")
print(f"Vessels delayed beyond desired departure: {metrics['vessels_delayed']}")

## 4. Visualize fitness, the schedule, and population diversity

The fitness curve shows the best fitness recorded in each generation; it is not a direct plot of service time. The schedule chart displays processing intervals by berth and dashed desired-departure markers. Population diversity is measured here as the fraction of distinct chromosomes in the population. It is a simple genotype-based indicator: high diversity means more distinct encoded schedules, not necessarily better schedules.

In [ ]:
def visualize_ga_results(
    ga_solver: GeneticAlgorithmDBAPSolver,
    vessels: List[Vessel],
    berths: List[Berth],
) -> None:
    """Plot search progress, the best schedule, metrics, and diversity."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat
    fig.suptitle("Genetic Algorithm Performance Analysis", fontsize=16)

    generations = np.arange(len(ga_solver.fitness_history))
    ax1.plot(generations, ga_solver.fitness_history, color="tab:blue", label="Best fitness")
    ax1.set_xlabel("Generation")
    ax1.set_ylabel("Fitness (dimensionless)")
    ax1.set_title("Best Fitness by Generation")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    solution = ga_solver.chromosome_to_solution(ga_solver.best_solution)
    solution_frame = pd.DataFrame(solution)
    vessel_colors = {
        vessel.id: plt.get_cmap("tab10")(index % 10)
        for index, vessel in enumerate(vessels)
    }

    for row in solution:
        ax2.barh(
            row["berth_id"],
            row["processing_time"],
            left=row["start_time"],
            color=vessel_colors[row["vessel_id"]],
            label=f"Vessel {row['vessel_id']}",
        )
        ax2.text(
            row["start_time"] + row["processing_time"] / 2,
            row["berth_id"],
            f"V{row['vessel_id']}",
            ha="center",
            va="center",
            color="white",
            fontweight="bold",
        )

    for vessel in vessels:
        ax2.axvline(
            vessel.desired_departure,
            color="red",
            linestyle="--",
            alpha=0.45,
            label="Desired departure" if vessel.id == vessels[0].id else None,
        )
    ax2.set_xlabel("Time (hours)")
    ax2.set_ylabel("Berth")
    ax2.set_title("Best Berth Schedule")
    ax2.set_yticks([berth.id for berth in berths])
    ax2.set_yticklabels([f"Berth {berth.id}" for berth in berths])
    ax2.grid(True, axis="x", alpha=0.3)
    handles, labels = ax2.get_legend_handles_labels()
    unique_handles = dict(zip(labels, handles))
    if unique_handles:
        ax2.legend(unique_handles.values(), unique_handles.keys(), loc="best")

    metrics = ga_solver.calculate_solution_metrics(solution)
    metric_names = [
        "Service time",
        "Waiting time",
        "Processing time",
        "Average waiting",
    ]
    metric_values = [
        metrics["total_service_time"],
        metrics["total_waiting_time"],
        metrics["total_processing_time"],
        metrics["avg_waiting_time"],
    ]
    bars = ax3.bar(metric_names, metric_values, color=["skyblue", "orange", "green", "salmon"])
    ax3.set_ylabel("Time (hours)")
    ax3.set_title("Schedule Metrics")
    ax3.tick_params(axis="x", rotation=25)
    for bar, value in zip(bars, metric_values):
        ax3.annotate(
            f"{value:.2f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
        )

    ax4.plot(
        generations,
        ga_solver.diversity_history,
        color="tab:red",
        marker="o",
        markersize=3,
    )
    ax4.set_xlabel("Generation")
    ax4.set_ylabel("Distinct chromosomes / population size")
    ax4.set_title("Genotype Diversity")
    ax4.set_ylim(0, 1.05)
    ax4.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()


visualize_ga_results(ga_solver, vessels, berths)

## 5. Compare with an earliest-deadline-first baseline

EDF processes vessels in order of desired departure and chooses, for each vessel, the compatible berth giving the earliest completion based on that berth's current availability. This is a simple deterministic baseline, not an exact optimizer. The comparison measures both schedules using the same service-time definitions. Because waiting time can be zero, percentage improvement is reported as not defined when the baseline value is zero; computation-time comparisons should be treated as rough observations rather than stable benchmarks.

In [ ]:
def compare_ga_vs_edf(
    ga_solver: GeneticAlgorithmDBAPSolver,
    vessels: List[Vessel],
    berths: List[Berth],
    ga_computation_time: float,
        ) -> None:
    """Compare one GA schedule with a simple EDF scheduling baseline."""
    print("GA vs EDF Heuristic Comparison")
    print("=" * 40)

    berth_availability = {berth.id: 0 for berth in berths}
    edf_solution = []
    for vessel in sorted(vessels, key=lambda item: item.desired_departure):
        feasible_options = []
        for berth in berths:
            if (
                vessel.length <= berth.capacity
                and berth.id in vessel.processing_times
            ):
                start = max(vessel.arrival_time, berth_availability[berth.id])
                completion = start + vessel.processing_times[berth.id]
                feasible_options.append((completion, berth.id, start))

        if not feasible_options:
            continue

        completion, berth_id, start = min(feasible_options)
        processing = vessel.processing_times[berth_id]
        waiting = start - vessel.arrival_time
        edf_solution.append(
            {
                "vessel_id": vessel.id,
                "berth_id": berth_id,
                "start_time": start,
                "completion_time": completion,
                "processing_time": processing,
                "waiting_time": waiting,
                "service_time": waiting + processing,
            }
        )
        berth_availability[berth_id] = completion

    def summarize(schedule):
        if not schedule:
            return {
                "total_service_time": 0,
                "total_waiting_time": 0,
                "total_processing_time": 0,
                "avg_waiting_time": 0,
            }
        frame = pd.DataFrame(schedule)
        return {
            "total_service_time": frame["service_time"].sum(),
            "total_waiting_time": frame["waiting_time"].sum(),
            "total_processing_time": frame["processing_time"].sum(),
            "avg_waiting_time": frame["waiting_time"].mean(),
        }

    edf_metrics = summarize(edf_solution)
    ga_solution = ga_solver.chromosome_to_solution(ga_solver.best_solution)
    ga_metrics = summarize(ga_solution)
    edf_metrics["computation_time"] = np.nan
    ga_metrics["computation_time"] = ga_computation_time

    metric_keys = [
        "total_service_time",
        "total_waiting_time",
        "total_processing_time",
        "avg_waiting_time",
        "computation_time",
    ]
    metric_names = [
        "Total service time (hours)",
        "Total waiting time (hours)",
        "Total processing time (hours)",
        "Average waiting time (hours)",
        "GA computation time (ms)",
    ]
    comparison_rows = []
    for key, name in zip(metric_keys, metric_names):
        edf_value = edf_metrics[key]
        ga_value = ga_metrics[key]
        if key == "computation_time":
            improvement = "N/A"
        elif edf_value == 0:
            improvement = "N/A"
        else:
            improvement = f"{(edf_value - ga_value) / edf_value * 100:+.1f}%"
        comparison_rows.append(
            {
                "Metric": name,
                "EDF": f"{edf_value:.2f}",
                "GA": f"{ga_value:.2f}",
                "GA improvement": improvement,
            }
        )

    comparison_frame = pd.DataFrame(comparison_rows)
    print(comparison_frame.to_string(index=False))

    plotted_keys = metric_keys[:4]
    x_positions = np.arange(len(plotted_keys))
    width = 0.35
    edf_values = [edf_metrics[key] for key in plotted_keys]
    ga_values = [ga_metrics[key] for key in plotted_keys]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.bar(x_positions - width / 2, edf_values, width, label="EDF", color="orange")
    ax1.bar(x_positions + width / 2, ga_values, width, label="GA", color="tab:blue")
    ax1.set_xlabel("Metric")
    ax1.set_ylabel("Time (hours)")
    ax1.set_title("Schedule Metrics: EDF and GA")
    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(
        ["Service", "Waiting", "Processing", "Average waiting"],
        rotation=25,
        ha="right",
    )
    ax1.legend()
    ax1.grid(True, axis="y", alpha=0.3)

    ax2.bar(["EDF", "GA"], [0, ga_computation_time], color=["orange", "tab:blue"])
    ax2.set_ylabel("Measured computation time (ms)")
    ax2.set_title("Timing Available for This Run")
    ax2.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

    print("\nInterpretation")
    print(
        "EDF is a single simple baseline and the GA is stochastic. "
        "This comparison describes these schedules only; it does not "
        "establish general dominance or optimality."
    )


compare_ga_vs_edf(ga_solver, vessels, berths, computation_time)

## 6. Key insights, strengths, and limitations

### What the algorithm is doing

- **Population-based search:** the algorithm keeps several candidate schedules rather than a single schedule.
- **Selection and elitism:** fitter candidates are more likely to become parents, and the best candidates are carried into the next generation.
- **Crossover and mutation:** crossover combines vessel ordering information; mutation can change one vessel's berth assignment. These operators create candidate variations, while repair restores feasible timing.
- **Stochastic behavior:** random initialization and genetic operators mean independent runs can return different schedules. Reproducible comparisons require a deliberately controlled random seed and repeated experiments; this notebook does not set a seed.

### How to read the results

A higher fitness is preferred by this implementation, but fitness combines an assignment bonus with a reciprocal cost term. Service time, waiting time, processing time, and deadline delays are the operational measures to inspect. A fitness curve that stops changing indicates that the best recorded score has not improved during those generations; it does not prove that the global optimum has been reached. Genotype diversity measures how many distinct chromosomes remain and should not be interpreted as solution quality by itself.

### Advantages and limitations

A GA can explore a discrete search space without requiring derivatives, and its representation and repair logic can be adapted to additional constraints. It is a heuristic, however: it has no optimality guarantee, its performance depends on the encoding and parameter choices, and its runtime and solution quality vary across instances and random runs. The quality of the answer depends especially on whether the chromosome, fitness function, and repair procedure represent the operational problem faithfully.

This example has only three vessels, so it cannot establish how the method scales to medium or large instances. The displayed EDF comparison is a baseline for context, not evidence that the GA is consistently better. More extensive evaluation would require appropriately sized instances and repeated runs, while keeping the model and objective fixed.

### Natural extensions

Within the same GA framework, the model could be extended with additional operational constraints or objective terms, provided the chromosome, fitness calculation, repair routine, and interpretation are updated consistently. Any such extension should distinguish hard feasibility requirements from soft preferences and state clearly how each is represented.